# PRISM-Phish: Full NVIDIA T4 GPU Training on Kaggle
Trains Transformer Baseline & PRISM-Phish Hybrid Model with GRL & Consistency on full 91,030 dataset.


In [ ]:
#!/usr/bin/env python3
"""
PRISM-Phish: Full GPU Training on Kaggle (NVIDIA Tesla T4)
Trains Transformer Baseline and PRISM-Phish on full 91,030 dataset using FP16 mixed precision.
"""

import os
import sys
import zipfile
import json
import time
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger("kaggle_gpu_train")

# 1. Setup Environment and Unzip src
logger.info("=== Setting up Kaggle GPU Environment ===")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
logger.info(f"Using Device: {device} ({gpu_name})")

DATA_DIR = Path("/kaggle/input/prism-phish-splits")
WORKING_DIR = Path("/kaggle/working")

# Unzip source code if present
src_zip = DATA_DIR / "src.zip"
if src_zip.exists():
    logger.info("Extracting src.zip to /kaggle/working/...")
    with zipfile.ZipFile(src_zip, "r") as z:
        z.extractall(WORKING_DIR)
sys.path.insert(0, str(WORKING_DIR))

from src.features.feature_pipeline import FeaturePipeline
from src.models.prism_phish import PRISMPhish
from src.evaluation.metrics import compute_all_metrics
from src.training.seeds import set_seed

set_seed(42)

# 2. Load Data Splits
logger.info("Loading data splits from Parquet...")
train_df = pd.read_parquet(DATA_DIR / "train.parquet")
val_df = pd.read_parquet(DATA_DIR / "val.parquet")
test_df = pd.read_parquet(DATA_DIR / "test.parquet")

logger.info(f"Loaded: Train={len(train_df):,}, Val={len(val_df):,}, Test={len(test_df):,}")


def prepare_text(df):
    s = df["subject"].fillna("").astype(str)
    b = df["body"].fillna("").astype(str)
    return (s + " " + b).tolist()


train_texts = prepare_text(train_df)
val_texts = prepare_text(val_df)
test_texts = prepare_text(test_df)

y_train = train_df["label"].values.astype(int)
y_val = val_df["label"].values.astype(int)
y_test = test_df["label"].values.astype(int)

# 3. Model Tokenizer
MODEL_NAME = "distilbert-base-uncased"
logger.info(f"Loading Tokenizer: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class PhishDataset(Dataset):
    def __init__(self, texts, struct_feats, labels, source_ids, tokenizer, max_len=256):
        self.encodings = tokenizer(texts, truncation=True, max_length=max_len, padding=False)
        self.struct_feats = torch.tensor(struct_feats, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.source_ids = torch.tensor(source_ids, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["struct_feats"] = self.struct_feats[idx]
        item["label"] = self.labels[idx]
        item["source_id"] = self.source_ids[idx]
        return item


def collate_fn(batch):
    input_ids = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
    attn_mask = [torch.tensor(b["attention_mask"], dtype=torch.long) for b in batch]
    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attn_mask = torch.nn.utils.rnn.pad_sequence(attn_mask, batch_first=True, padding_value=0)
    struct_feats = torch.stack([b["struct_feats"] for b in batch])
    labels = torch.stack([b["label"] for b in batch])
    source_ids = torch.stack([b["source_id"] for b in batch])
    return {
        "input_ids": input_ids,
        "attention_mask": attn_mask,
        "struct_feats": struct_feats,
        "labels": labels,
        "source_ids": source_ids,
    }


# 4. Extract Structural Features
logger.info("Extracting 52 structural and security features via FeaturePipeline...")
feat_pipe = FeaturePipeline(normalization="standard")
X_train_struct = feat_pipe.fit_transform(train_df)
X_val_struct = feat_pipe.transform(val_df)
X_test_struct = feat_pipe.transform(test_df)

unique_sources = sorted(list(train_df["dataset_id"].unique()))
source_map = {s: i for i, s in enumerate(unique_sources)}
src_train = [source_map.get(s, 0) for s in train_df["dataset_id"]]
src_val = [source_map.get(s, 0) for s in val_df["dataset_id"]]
src_test = [source_map.get(s, 0) for s in test_df["dataset_id"]]

train_dataset = PhishDataset(train_texts, X_train_struct, y_train, src_train, tokenizer)
val_dataset = PhishDataset(val_texts, X_val_struct, y_val, src_val, tokenizer)
test_dataset = PhishDataset(test_texts, X_test_struct, y_test, src_test, tokenizer)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

# 5. Initialize PRISM-Phish Model
logger.info(f"Initializing PRISM-Phish ({X_train_struct.shape[1]} features, {len(unique_sources)} sources)...")
model = PRISMPhish(
    model_name=MODEL_NAME,
    num_structural_features=X_train_struct.shape[1],
    num_sources=len(unique_sources),
    domain_adversarial=True,
    grl_lambda_max=0.5,
)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
EPOCHS = 3
total_steps = len(train_loader) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

logger.info(f"=== Starting Full GPU Training ({EPOCHS} Epochs, {len(train_loader)} steps/epoch) ===")

best_val_auc = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    start = time.time()
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attn_mask = batch["attention_mask"].to(device)
        struct_feats = batch["struct_feats"].to(device)
        labels = batch["labels"].to(device)
        source_ids = batch["source_ids"].to(device)

        current_step = step + (epoch - 1) * len(train_loader)
        model.update_grl_lambda(current_step, total_steps)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attn_mask,
                structural_features=struct_feats,
                source_labels=source_ids,
                labels=labels,
            )
            loss = outputs["phishing_loss"]
            if "domain_loss" in outputs and outputs["domain_loss"] is not None:
                loss = loss + 0.2 * outputs["domain_loss"]

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        if (step + 1) % 250 == 0 or (step + 1) == len(train_loader):
            elapsed = time.time() - start
            logger.info(f"Epoch {epoch}/{EPOCHS} | Step {step+1}/{len(train_loader)} | Loss: {total_loss/(step+1):.4f} | Elapsed: {elapsed:.1f}s")

    # Validation
    model.eval()
    val_probs, val_preds, val_targets = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                out = model(
                    input_ids=batch["input_ids"].to(device),
                    attention_mask=batch["attention_mask"].to(device),
                    structural_features=batch["struct_feats"].to(device),
                )
            p = torch.softmax(out["logits"], dim=-1)[:, 1]
            val_probs.extend(p.cpu().numpy())
            val_preds.extend(torch.argmax(out["logits"], dim=-1).cpu().numpy())
            val_targets.extend(batch["labels"].numpy())

    val_metrics = compute_all_metrics(np.array(val_targets), np.array(val_preds), np.array(val_probs), prefix="val")
    logger.info(f"Epoch {epoch} Complete -> Val ROC-AUC: {val_metrics['val_roc_auc']:.4f}, Val F1: {val_metrics['val_f1']:.4f}")

    if val_metrics["val_roc_auc"] > best_val_auc:
        best_val_auc = val_metrics["val_roc_auc"]
        torch.save(model.state_dict(), WORKING_DIR / "best_prism_phish_gpu.pt")
        logger.info(f"Saved new best model checkpoint! Val ROC-AUC: {best_val_auc:.4f}")

# 6. Final Test Set Evaluation
logger.info("=== Final Evaluation on Held-Out Test Set (19,052 samples) ===")
model.eval()
test_probs, test_preds, test_targets = [], [], []
with torch.no_grad():
    for batch in test_loader:
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            out = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                structural_features=batch["struct_feats"].to(device),
            )
        p = torch.softmax(out["logits"], dim=-1)[:, 1]
        test_probs.extend(p.cpu().numpy())
        test_preds.extend(torch.argmax(out["logits"], dim=-1).cpu().numpy())
        test_targets.extend(batch["labels"].numpy())

test_metrics = compute_all_metrics(np.array(test_targets), np.array(test_preds), np.array(test_probs), prefix="test")
logger.info(
    f"Final Test ROC-AUC: {test_metrics['test_roc_auc']:.4f}, Test PR-AUC: {test_metrics['test_pr_auc']:.4f}, "
    f"Test F1: {test_metrics['test_f1']:.4f}, Test Accuracy: {test_metrics['test_accuracy']*100:.2f}%"
)

# Save outputs to /kaggle/working/
out_file = WORKING_DIR / "prism_phish_gpu_results.json"
with open(out_file, "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2)
logger.info(f"Results successfully saved to {out_file}!")
